In [5]:
import numpy as np
import pandas as pd
from pyscenic.cli.utils import (
    ATTRIBUTE_NAME_CELL_IDENTIFIER,
    ATTRIBUTE_NAME_GENE,
    append_auc_mtx,
    is_valid_suffix,
    load_adjacencies,
    load_exp_matrix,
    load_modules,
    load_signatures,
    save_enriched_motifs,
    save_matrix,
    suffixes_to_separator,
)

In [ ]:
from pyscenic.utils import add_correlation, modules_from_adjacencies

In [3]:
pdir = "/dcs04/lieber/marmaypag/spatialDLPFC_mdd_bpd_LIBD4100/spatialDLPFC_mdd_bpd/"

In [12]:
adjacencies = load_adjacencies(pdir+"processed-data/09_SCENIC/spe-n119_21077_adj_with-corr.csv")
adjacencies.shape

In [13]:
adjacencies.head()

,TF,target,importance,regulation,rho
0,PRNP,GPM6A,41.904825,1,0.6890393220321283
1,PRNP,CHN1,39.875782,1,0.7033438801318207
2,MYRF,PLP1,38.896794,1,0.5970946738999354
3,PRNP,CALM1,38.549526,1,0.7179264885218953
4,PRNP,CPE,37.955888,1,0.6147579443825747


In [21]:
laDEGs = pd.read_csv(pdir+"processed-data/07_dx_DE/layer-adjusted-pc3-age-nspots_smoothed-k9-1663_dx-sex_degs-F-test-t-test.csv")
lrDEGs = pd.read_csv(pdir+"processed-data/07_dx_DE/layer-restricted-pc3-age-nspots_smoothed-k9-1663_dx-sex_degs-F-test-t-test.csv")

In [31]:
mbvDEGs = list(set(pd.concat([laDEGs.gene_name, lrDEGs.gene_name])))
len(mbvDEGs)

649

In [23]:
mbvDEGs = list(set(list([laDEGs.gene_name, lrDEGs.gene_name])))
len(mbvDEGs)

TypeError: unhashable type: 'Series'

In [ ]:
signature_column_names = list(laDEGs.gene_name.values)

In [32]:
adjacencies2 = adjacencies[adjacencies.TF.isin(mbvDEGs)]

In [33]:
adjacencies2.shape

(105796, 5)

In [8]:
from functools import partial
from itertools import chain
from typing import Sequence, Type
from urllib.parse import urljoin

import numpy as np
import pandas as pd
from ctxcore.genesig import GeneSignature, Regulon, openfile

from pyscenic.utils import modules4thr, modules4top_targets, modules4top_factors, _create_idx_pairs

COLUMN_NAME_TF = "TF"
COLUMN_NAME_MOTIF_ID = "MotifID"
COLUMN_NAME_MOTIF_SIMILARITY_QVALUE = "MotifSimilarityQvalue"
COLUMN_NAME_ORTHOLOGOUS_IDENTITY = "OrthologousIdentity"
COLUMN_NAME_ANNOTATION = "Annotation"

COLUMN_NAME_TARGET = "target"
COLUMN_NAME_WEIGHT = "importance"
COLUMN_NAME_REGULATION = "regulation"
COLUMN_NAME_CORRELATION = "rho"
RHO_THRESHOLD = 0.03

In [35]:
def select_modules_from_adjacencies(
    adjacencies: pd.DataFrame,
#    ex_mtx: pd.DataFrame,
    thresholds=(0.75, 0.90),
    top_n_targets=(50,),
    top_n_regulators=(5, 10, 50),
    min_genes=20,
    absolute_thresholds=True,
    keep_only_activating=False
) -> Sequence[Regulon]:
    # To make the pySCENIC code more robust to the selection of the network inference method in the first step of
    # the pipeline, it is better to use percentiles instead of absolute values for the weight thresholds.
    if not absolute_thresholds:

        def iter_modules(adjc, context):
            yield from chain(
                chain.from_iterable(
                    modules4thr(
                        adjc, thr, context, pattern="weight>{}%".format(frac * 100)
                    )
                    for thr, frac in zip(
                        list(adjacencies[COLUMN_NAME_WEIGHT].quantile(thresholds)),
                        thresholds,
                    )
                ),
                chain.from_iterable(
                    modules4top_targets(adjc, n, context) for n in top_n_targets
                ),
                chain.from_iterable(
                    modules4top_factors(adjc, n, context) for n in top_n_regulators
                ),
            )

    else:

        def iter_modules(adjc, context):
            yield from chain(
                chain.from_iterable(
                    modules4thr(adjc, thr, context) for thr in thresholds
                ),
                chain.from_iterable(
                    modules4top_targets(adjc, n, context) for n in top_n_targets
                ),
                chain.from_iterable(
                    modules4top_factors(adjc, n, context) for n in top_n_regulators
                ),
            )
    
    
    activating_modules = adjacencies[adjacencies['regulation'] > 0.0]
    if keep_only_activating:
        modules_iter = iter_modules(
            activating_modules, frozenset(['activating'])
        )
    else:
        repressing_modules = adjacencies[adjacencies['regulation'] < 0.0]
        modules_iter = chain(
            iter_modules(activating_modules, frozenset(['activating'])),
            iter_modules(repressing_modules, frozenset(['repressing'])),
        )

    # Derive modules for these adjacencies.
    # + Add the transcription factor to the module.
    #   [We are unable to assess if a TF works in a direct self-regulating way, either inhibiting its own expression or
    #    activating it. Therefore the most unbiased way forward is to add the TF to both activating as well as
    #    repressing modules]
    # + Filter for minimum number of genes.
    #LOGGER.info("Creating modules.")

    def add_tf(module):
        return module.add(module.transcription_factor)

    return list(filter(lambda m: len(m) >= min_genes, map(add_tf, modules_iter)))


In [39]:
type(adjacencies2)

pandas.core.frame.DataFrame

In [42]:
type((0.75,))

tuple

In [43]:
modules2 = select_modules_from_adjacencies(adjacencies2, thresholds=(0.5,))

In [65]:
modules2[1].transcription_factor

'ARID5A'

In [70]:
len(modules2[1].genes)

50

In [81]:
[next(iter(item.context)) for item in modules2]

['activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activating',
 'activati

In [102]:
mod_df = pd.DataFrame({"TF": [item.transcription_factor for item in modules2], "set_size": [len(item) for item in modules2]})

In [103]:
mod_df[mod_df.TF=="ELK1"]

,TF,set_size
12,ELK1,51
68,ELK1,51
125,ELK1,322
182,ELK1,350
239,ELK1,351
289,ELK1,26
321,ELK1,51
378,ELK1,334
435,ELK1,335
492,ELK1,335
